In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
from functools import partial
import shutil
import sys
import json

import ase.io
from ase.atoms import Atoms
from ase.visualize import view
import numpy as onp

from msmjax.calculators import set_up_msm_params
import msmjax

parentpath_charged_lj_utils = str(
    Path(msmjax.__file__).resolve().parents[2] / "examples" / "charged_lj_md"
)
sys.path.append(parentpath_charged_lj_utils)
path_charged_lj_utils = parentpath_charged_lj_utils / "utils_charged_lj"

from utils_charged_lj.energy_model import (
    EPSILON_ELECTRONVOLT,
    SIGMA_ANGSTROM,
    make_charged_lj_evaluation_fns,
    GenericWrapperCalculator,
)
from utils_charged_lj.helpers import make_convert_lj_to_ase, make_md_command

from clinamen2.utils.structure_setup import place_atoms_packmol

# Definitions

In [2]:
PACKMOL_EXECUTABLE = "/home/florian/Downloads/packmol-20.14.2/packmol"

In [3]:
def generate_initial_structure(
    n_particles: int,
    density: float,
    tolerance: float,
    packmol_executable: str,
    seed=None,
):
    N_DIM = 3

    side_length = (n_particles / density) ** (1 / N_DIM)
    cell = onp.eye(N_DIM) * side_length
    pbc = onp.ones(N_DIM, dtype=bool)

    # If the average particle spacing at a given density is less than the
    # tolerance parameter, reduce tolerance to the average particle spacing,
    # since otherwise no packing solution is possible (and packmol will waste
    # time trying).
    avg_particle_spacing = 1 / density ** (1 / N_DIM)
    tolerance = min(tolerance, avg_particle_spacing)

    positions = place_atoms_packmol(
        n_atoms=n_particles,
        side_length=side_length - tolerance,  # small gap volume for PBCs
        tolerance=tolerance,
        exec_string=packmol_executable,
        random_seed=-1 if seed is None else seed,
    )
    atoms = Atoms(
        cell=cell,
        pbc=pbc,
        positions=positions,
        symbols=["Ar"] * n_particles,
    )
    # Atoms in packmol's solution can end up very slightly outside the cell.
    # Wrap back any such atoms.
    atoms.wrap()

    rng = onp.random.default_rng(seed=seed)
    charges = onp.ones(n_particles)
    charges[
        rng.choice(n_particles, size=n_particles // 2, replace=False)
    ] *= -1.0
    atoms.set_initial_charges(charges)

    return atoms

In [4]:
# Parameters defining the position in the phase diagram (in Lennard-Jones units)
TEMPERATURE_LJ = 3.15
DENSITY_LJ = 0.85

# Parameters for structure generation
N_PARTICLES = 1000
TOLERANCE = 2 ** (1 / 6) * SIGMA_ANGSTROM  # location of LJ potential minimum

SEED = 3456

In [5]:
convert_lj_to_ase = make_convert_lj_to_ase(
    ref_epsilon=EPSILON_ELECTRONVOLT, ref_sigma=SIGMA_ANGSTROM
)
phaseparams_ase_units = convert_lj_to_ase(
    density=DENSITY_LJ, temperature=TEMPERATURE_LJ
)

In [6]:
# Ratio between the Coulomb energy and the Lennard-Jones energy scale epsilon
# at a distance equal to the Lennard-Jones distance scale sigma:
COULOMB_STRENGTH = 50.0

# Shared cutoff used both for LJ potential and short-range part MSM
CUTOFF = 4.0 * SIGMA_ANGSTROM
P = 4
PBC = (True,) * 3

CELL_MODE = "ortho"
# In NVE and NVT, dynamic_cell is technically not needed, but we still use it
# in order to be able to on-the-fly evaluate and record the pressure along the
# trajectory.
# If performance matters, one could first run without dynamic cell and then
# evaluate the pressure for the saved snapshots afterwards.
DYNAMIC_CELL = True

# Generate structure

In [7]:
structure = generate_initial_structure(
    n_particles=N_PARTICLES,
    density=phaseparams_ase_units["density"],
    tolerance=TOLERANCE,
    packmol_executable=PACKMOL_EXECUTABLE,
    seed=SEED,
)

# We implement the Coulomb/Lennard-Jones potential strength ratio by setting
# the magnitude of the particle charges to an according effective value
# (instead of introducing an additional prefactor for the Coulomb term of the
# total potential energy, which would be the other option)
effective_abs_charge = onp.sqrt(
    COULOMB_STRENGTH * SIGMA_ANGSTROM * EPSILON_ELECTRONVOLT
)
structure.set_initial_charges(
    effective_abs_charge * onp.sign(structure.get_initial_charges())
)

tolerance 3.5312067571664123
filetype xyz 
output box_1000.xyz
seed 3456

structure trivial.xyz
  number 1000
  inside box 0. 0. 0. 31.78086081449771 31.78086081449771 31.78086081449771
end structure


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                              Version 20.14.2 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: xyz
  Seed for random number generator:         3456
  Output file: box_1000.xyz
  Reading coordinate file: trivial.xyz
  Number of independent structures:            1
  The structures are: 
  Structure            1 :sphere(           1  

In [8]:
avg_interparticle_distance = (1 / phaseparams_ase_units["density"]) ** (1 / 3)
msm_params = set_up_msm_params(
    cell=onp.asarray(structure.get_cell()),
    level_one_spacings=min(avg_interparticle_distance, SIGMA_ANGSTROM),
    level_zero_cutoff=CUTOFF,
    p=P,
    pbc=PBC,
    cell_mode=CELL_MODE,
    dynamic_cell=DYNAMIC_CELL,
)
evaluation_fns = make_charged_lj_evaluation_fns(
    msm_params=msm_params, sigma=SIGMA_ANGSTROM, epsilon=EPSILON_ELECTRONVOLT
)

In [9]:
from ase.optimize import BFGSLineSearch

effective_charges = structure.get_initial_charges()
structure.calc = GenericWrapperCalculator(
    energy_fn=partial(evaluation_fns["energy"], charges=effective_charges),
    forces_fn=partial(evaluation_fns["forces"], charges=effective_charges),
    stress_fn=partial(evaluation_fns["stress"], charges=effective_charges),
)
print("- Running energy minimization on initial structure.")
dyn = BFGSLineSearch(structure)
dyn.run(fmax=0.1, steps=200)
structure.wrap()

view(structure)

- Running energy minimization on initial structure.
                Step[ FC]     Time          Energy          fmax
BFGSLineSearch:    0[  0] 12:36:06     -123.190293       1.2423
BFGSLineSearch:    1[  4] 12:36:06     -244.229825       8.7579
BFGSLineSearch:    2[  6] 12:36:07     -258.708011       1.6193
BFGSLineSearch:    3[  8] 12:36:08     -287.185506       3.9228
BFGSLineSearch:    4[ 10] 12:36:09     -309.642297       1.5952
BFGSLineSearch:    5[ 12] 12:36:10     -325.762397       2.3497
BFGSLineSearch:    6[ 14] 12:36:11     -342.448818       4.8990
BFGSLineSearch:    7[ 16] 12:36:12     -354.131386       1.3150
BFGSLineSearch:    8[ 18] 12:36:13     -368.858630       3.4517
BFGSLineSearch:    9[ 19] 12:36:14     -376.038726       1.0252
BFGSLineSearch:   10[ 21] 12:36:15     -387.085959       1.5845
BFGSLineSearch:   11[ 22] 12:36:16     -394.997903       0.5657
BFGSLineSearch:   12[ 24] 12:36:17     -402.180973       2.0510
BFGSLineSearch:   13[ 25] 12:36:18     -407.108698 

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

In [10]:
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
)

rng = onp.random.default_rng(SEED)
MaxwellBoltzmannDistribution(
    structure, temperature_K=2 * phaseparams_ase_units["temperature"], rng=rng
)
Stationary(structure)

Error processing line 1 of /home/florian/anaconda3/envs/msmjax_py3.11/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 195, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored


# Write the input files for the simulations

In [11]:
FILENAME_MD_SCRIPT = "run_md_ase.py"
FILENAME_INPUT_STRUCT = "initial_structure.xyz"
FILENAME_MSM_PARAMS = "msm_params.json"
DIRNAME_OUT = "out/"

## NVE

In [12]:
ensemble = "NVE"
rundir = Path(ensemble + "/")
simtime_ps = 1000.0
timestep_fs = 1.0
loginterval_fs = 500.0

md_setup_info = {
    "simtime_ps": simtime_ps,
    "timestep_fs": timestep_fs,
    "loginterval_fs": loginterval_fs,
    "ensemble": ensemble,
}

rundir.mkdir()

ase.io.write(rundir / FILENAME_INPUT_STRUCT, structure)
msm_params.save_json(rundir / FILENAME_MSM_PARAMS, indent=2)
with open(rundir / "md_setup_info.json", "w") as f:
    json.dump(md_setup_info, f, indent=2)
shutil.copy(path_charged_lj_utils / FILENAME_MD_SCRIPT, rundir)
command = make_md_command(
    simtime_ps=simtime_ps,
    timestep_fs=timestep_fs,
    loginterval_fs=loginterval_fs,
    filename_md_script=FILENAME_MD_SCRIPT,
    filename_input_struct=FILENAME_INPUT_STRUCT,
    filename_msm_params=FILENAME_MSM_PARAMS,
    dirname_out=DIRNAME_OUT,
    ensemble=ensemble,
)
with open(rundir / "runscript.sh", "w") as f:
    f.write(command)

## NVT

In [13]:
ensemble = "NVT"
rundir = Path(ensemble + "/")
simtime_ps = 1000.0
timestep_fs = 1.0
loginterval_fs = 500.0

md_setup_info = {
    "simtime_ps": simtime_ps,
    "timestep_fs": timestep_fs,
    "loginterval_fs": loginterval_fs,
    "ensemble": ensemble,
    "temp_K": phaseparams_ase_units["temperature"],
}

rundir.mkdir()

ase.io.write(rundir / FILENAME_INPUT_STRUCT, structure)
msm_params.save_json(rundir / FILENAME_MSM_PARAMS, indent=2)
with open(rundir / "md_setup_info.json", "w") as f:
    json.dump(md_setup_info, f, indent=2)
shutil.copy(path_charged_lj_utils / FILENAME_MD_SCRIPT, rundir)
command = make_md_command(
    simtime_ps=simtime_ps,
    timestep_fs=timestep_fs,
    loginterval_fs=loginterval_fs,
    filename_md_script=FILENAME_MD_SCRIPT,
    filename_input_struct=FILENAME_INPUT_STRUCT,
    filename_msm_params=FILENAME_MSM_PARAMS,
    dirname_out=DIRNAME_OUT,
    ensemble=ensemble,
    temp_K=phaseparams_ase_units["temperature"],
)
with open(rundir / "runscript.sh", "w") as f:
    f.write(command)